In [1]:
import numpy as np
import pytest
import h5netcdf
from h5netcdf import legacyapi
import h5py
import netCDF4

# NetCDF4 DIMENSIONS

## Problems

### h5netcdf

- dimension incorrectly identified as variable

#### create test data with NetCDF4

In [13]:
tmp_local_or_remote_netcdf = "test_dim_var_netcdf4.nc"
print("----------------------------------------")
with netCDF4.Dataset(tmp_local_or_remote_netcdf, "w") as f:
    f.createDimension("time", 0) # time
    f.createDimension("nvec", 1) # nvec
    f.createDimension("sample", 2) # sample
    f.createDimension("ship", 3) # ship
    f.createDimension("ship_strlen", 10) # ship_strlen
    
print("----------------------------------------")
with netCDF4.Dataset(tmp_local_or_remote_netcdf, "a") as f:
    f.createVariable("time", "f8", ("time",))
    f.createVariable("data", "i8", ("ship", "sample", "time", "nvec", ))
    f.createVariable("sample", "i8", ("time", "sample",)) 
    f.createVariable("ship", "S1", ("ship", "ship_strlen",)) 
    #f["y"][:] = np.ones((3,1))
    
    
# print("----------------------------------------")
# with h5netcdf.File(tmp_local_or_remote_netcdf, "r") as f:
#     print("*********************************************")
#     print("dims:", f.dimensions)
#     print("vars:", list(f.variables))
#     print("time:", f.variables["time"])
#     print("sample:", f.variables["sample"])
#     print("ship:", f.variables["ship"])
#     print("data:", f.variables["data"])

#with h5netcdf.File(tmp_local_or_remote_netcdf, "a") as f:
#    f.attrs["test"] = 1
    
    
    #print("y:", f.variables["_nc4_non_coord_y"])
    #print("y:", f.variables["y"])

----------------------------------------
----------------------------------------


In [14]:
!ncdump test_dim_var_netcdf4.nc

netcdf test_dim_var_netcdf4 {
dimensions:
	time = UNLIMITED ; // (0 currently)
	nvec = 1 ;
	sample = 2 ;
	ship = 3 ;
	ship_strlen = 10 ;
variables:
	double time(time) ;
	int64 data(ship, sample, time, nvec) ;
	int64 sample(time, sample) ;
	char ship(ship, ship_strlen) ;
data:

 ship =
  "",
  "",
  "" ;
}


In [15]:
!h5dump test_dim_var_netcdf4.nc

HDF5 "test_dim_var_netcdf4.nc" {
GROUP "/" {
   ATTRIBUTE "_NCProperties" {
      DATATYPE  H5T_STRING {
         STRSIZE 34;
         STRPAD H5T_STR_NULLTERM;
         CSET H5T_CSET_ASCII;
         CTYPE H5T_C_S1;
      }
      DATASPACE  SCALAR
      DATA {
      (0): "version=2,netcdf=4.7.4,hdf5=1.10.6"
      }
   }
   DATASET "_nc4_non_coord_sample" {
      DATATYPE  H5T_STD_I64LE
      DATASPACE  SIMPLE { ( 0, 2 ) / ( H5S_UNLIMITED, 2 ) }
      DATA {
      }
      ATTRIBUTE "DIMENSION_LIST" {
         DATATYPE  H5T_VLEN { H5T_REFERENCE { H5T_STD_REF_OBJECT }}
         DATASPACE  SIMPLE { ( 2 ) / ( 2 ) }
         DATA {
         (0): (DATASET 1394 /time ), (DATASET 1040 /sample )
         }
      }
      ATTRIBUTE "_Netcdf4Coordinates" {
         DATATYPE  H5T_STD_I32LE
         DATASPACE  SIMPLE { ( 2 ) / ( 2 ) }
         DATA {
         (0): 0, 2
         }
      }
   }
   DATASET "data" {
      DATATYPE  H5T_STD_I64LE
      DATASPACE  SIMPLE { ( 3, 2, 0, 1 ) / ( 3, 2, H5S_UNLIM

## A - dimension with no coordinate variable

```python
f.createDimension("nvec", 1)
f.createDimension("ship_strlen", 10)
```

```json
DATASET "nvec":
      ATTRIBUTE "CLASS":
         (0): "DIMENSION_SCALE"
      ATTRIBUTE "NAME":
         (0): "This is a netCDF dimension but not a netCDF variable.         1"
      ATTRIBUTE "REFERENCE_LIST":
         (0): { DATASET 331 /data , 3 }
      ATTRIBUTE "_Netcdf4Dimid":
         (0): 1
DATASET "ship_strlen":
      ATTRIBUTE "CLASS"
         (0): "DIMENSION_SCALE"
      ATTRIBUTE "NAME":
         (0): "This is a netCDF dimension but not a netCDF variable.        10"
      ATTRIBUTE "_Netcdf4Dimid":
         (0): 4
```

## B - dimension and coordinate variable 1D

```python
f.createDimension("time", 0)
f.createVariable("time", "f8", ("time",))
```

```json
DATASET "time":
      ATTRIBUTE "CLASS":
         (0): "DIMENSION_SCALE"
      ATTRIBUTE "NAME":
         (0): "time"
      ATTRIBUTE "REFERENCE_LIST":
         (0): { DATASET 331 /data , 2 },
         (1): { DATASET 2248 /_nc4_non_coord_sample , 0 }
      ATTRIBUTE "_Netcdf4Dimid":
         (0): 0
```

## C - dimension and data variable
```python
f.createDimension("sample", 2)
f.createVariable("sample", "i8", ("time", "sample",)) 
```

```json
DATASET "_nc4_non_coord_sample":
      ATTRIBUTE "DIMENSION_LIST": 
        (0): (DATASET 1394 /time ), (DATASET 1040 /sample )
      ATTRIBUTE "_Netcdf4Coordinates": 
        (0): 0, 2
DATASET "sample":
      ATTRIBUTE "CLASS":
         (0): "DIMENSION_SCALE"
      ATTRIBUTE "NAME":
         (0): "This is a netCDF dimension but not a netCDF variable.         2"
      ATTRIBUTE "REFERENCE_LIST": {
         (0): { DATASET 331 /data , 1 },
         (1): { DATASET 2248 /_nc4_non_coord_sample , 1 }
      ATTRIBUTE "_Netcdf4Dimid":
         (0): 2
```

## D - dimension and  coordinate variable 2D
```python
f.createDimension("ship", 3)
f.createVariable("ship", "S1", ("ship", "ship_strlen",)) 
```

```json
DATASET "ship": {
      ATTRIBUTE "CLASS":
         (0): "DIMENSION_SCALE"
      ATTRIBUTE "NAME":
         (0): "ship"
      ATTRIBUTE "REFERENCE_LIST":
         (0): { DATASET 331 /data , 0 }
      ATTRIBUTE "_Netcdf4Coordinates":
         (0): 3, 4
      ATTRIBUTE "_Netcdf4Dimid":
         (0): 3
```

## E - data variable
```python
f.createVariable("data", "i8", ("ship", "sample", "time", "nvec", ))
```

```json
DATASET "data":
      ATTRIBUTE "DIMENSION_LIST":
         (0): (DATASET 2570 /ship ), (DATASET 1040 /sample ),
         (2): (DATASET 1394 /time ), (DATASET 686 /nvec )
      ATTRIBUTE "_Netcdf4Coordinates":
         (0): 3, 2, 0, 1
```

### create anlalog with h5netcdf

In [9]:
tmp_local_or_remote_netcdf = "test_dim_var_legacy.nc"
print("----------------------------------------")
with legacyapi.Dataset(tmp_local_or_remote_netcdf, "w") as f:
    f.createDimension("time", 0) # time
    f.createDimension("nvec", 1) # nvec
    f.createDimension("sample", 2) # sample
    f.createDimension("ship", 3) # ship
    f.createDimension("ship_strlen", 10) # ship_strlen
    
print("----------------------------------------")
with legacyapi.Dataset(tmp_local_or_remote_netcdf, "a") as f:
    f.createVariable("time", "f8", ("time",))
    f.createVariable("data", "i8", ("ship", "sample", "time", "nvec", ))
    f.createVariable("sample", "i8", ("time", "sample",)) 
    f.createVariable("ship", "S1", ("ship", "ship_strlen",)) 
    #f["y"][:] = np.ones((3,1))
    
    
# print("----------------------------------------")
# with h5netcdf.File(tmp_local_or_remote_netcdf, "r") as f:
#     print("*********************************************")
#     print("dims:", f.dimensions)
#     print("vars:", list(f.variables))
#     print("time:", f.variables["time"])
#     print("sample:", f.variables["sample"])
#     print("ship:", f.variables["ship"])
#     print("data:", f.variables["data"])

#with h5netcdf.File(tmp_local_or_remote_netcdf, "a") as f:
#    f.attrs["test"] = 1
    
    
    #print("y:", f.variables["_nc4_non_coord_y"])
    #print("y:", f.variables["y"])

----------------------------------------
----------------------------------------
variable or dim scale: nvec <HDF5 dataset "nvec": shape (1,), type ">f4">
attrs: nvec [('CLASS', b'DIMENSION_SCALE'), ('NAME', b'This is a netCDF dimension but not a netCDF variable.         1'), ('_Netcdf4Dimid', 1)]
variable or dim scale: sample <HDF5 dataset "sample": shape (2,), type ">f4">
attrs: sample [('CLASS', b'DIMENSION_SCALE'), ('NAME', b'This is a netCDF dimension but not a netCDF variable.         2'), ('_Netcdf4Dimid', 2)]
variable or dim scale: ship <HDF5 dataset "ship": shape (3,), type ">f4">
attrs: ship [('CLASS', b'DIMENSION_SCALE'), ('NAME', b'This is a netCDF dimension but not a netCDF variable.         3'), ('_Netcdf4Dimid', 3)]
variable or dim scale: ship_strlen <HDF5 dataset "ship_strlen": shape (10,), type ">f4">
attrs: ship_strlen [('CLASS', b'DIMENSION_SCALE'), ('NAME', b'This is a netCDF dimension but not a netCDF variable.        10'), ('_Netcdf4Dimid', 4)]
variable or dim sc

TypeError: data type 'd8' not understood

In [6]:
!ncdump test_dim_var_legacy.nc

netcdf test_dim_var_legacy {
dimensions:
	time = UNLIMITED ; // (0 currently)
	nvec = 1 ;
	sample = 2 ;
	ship = 3 ;
	ship_strlen = 10 ;
variables:
	int64 data(ship, sample, time, nvec) ;
	float sample(sample) ;
	float ship(ship) ;
	float time(time) ;
data:

 sample = 0, 0 ;

 ship = 0, 0, 0 ;
}


In [7]:
!h5dump test_dim_var_legacy.nc

HDF5 "test_dim_var_legacy.nc" {
GROUP "/" {
   DATASET "data" {
      DATATYPE  H5T_STD_I64LE
      DATASPACE  SIMPLE { ( 3, 2, 0, 1 ) / ( 3, 2, 0, 1 ) }
      DATA {
      }
      ATTRIBUTE "DIMENSION_LIST" {
         DATATYPE  H5T_VLEN { H5T_REFERENCE { H5T_STD_REF_OBJECT }}
         DATASPACE  SIMPLE { ( 4 ) / ( 4 ) }
         DATA {
         (0): (DATASET 2272 /ship ), (DATASET 1904 /sample ),
         (2): (DATASET 800 /time ), (DATASET 1536 /nvec )
         }
      }
   }
   DATASET "nvec" {
      DATATYPE  H5T_IEEE_F32BE
      DATASPACE  SIMPLE { ( 1 ) / ( 1 ) }
      DATA {
      (0): 0
      }
      ATTRIBUTE "CLASS" {
         DATATYPE  H5T_STRING {
            STRSIZE 16;
            STRPAD H5T_STR_NULLTERM;
            CSET H5T_CSET_ASCII;
            CTYPE H5T_C_S1;
         }
         DATASPACE  SCALAR
         DATA {
         (0): "DIMENSION_SCALE"
         }
      }
      ATTRIBUTE "NAME" {
         DATATYPE  H5T_STRING {
            STRSIZE 64;
            STRPAD H5T_S

In [8]:
assert 1 == 2

AssertionError: 

In [ ]:
NOT_A_VARIABLE = b"This is a netCDF dimension but not a netCDF variable."
tmp_local_or_remote_netcdf = "test_dim_var_mismatch.h5"
with h5netcdf.File(tmp_local_or_remote_netcdf, "w") as f:
    f.dimensions["x"] = 2
    f.dimensions["z"] = 2
    f.create_variable("y", data=[1, 2], dimensions=("x",))

# with h5py.File(tmp_local_or_remote_netcdf, "r") as f:
#     dimlen = f"{f['y'].dims[0].values()[0].size:10}"
#     assert f["y"].dims[0].keys() == [NOT_A_VARIABLE.decode("ascii") + dimlen]
print("------------------------------------------------------------")
with h5netcdf.File(tmp_local_or_remote_netcdf, "a") as f:
    print(f.dimensions)
    print(list(f.variables))
    f.create_variable("x", data=[0, 1], dimensions=("z",))
    print(f.dimensions)
    print(list(f.variables))
    print(f.variables["x"])
print("------------------------------------------------------------")
with h5netcdf.File(tmp_local_or_remote_netcdf, "r") as f:
    print(f.dimensions)
    print(list(f.variables))
    for k, v in f.variables.items():
        print(k, v)
    #assert f["y"].dims[0].keys() == ["x"]
    #assert f["x"].dims[0].keys() == ["z"]


# with h5py.File(tmp_local_or_remote_netcdf, "r") as f:
#     assert f["y"].dims[0].keys() == ["x"]
#     assert f["x"].dims[0].keys() == ["z"]

In [ ]:
!ncdump test_dim_var_mismatch.h5

In [ ]:
!h5dump test_dim_var_mismatch.h5

In [ ]:
tmp_local_or_remote_netcdf = "test_dims_netcdf4.nc"
with netCDF4.Dataset(tmp_local_or_remote_netcdf, "w") as f:
    f.createDimension("x", 0)
    f.createDimension("y", 1)
    f.createDimension("z", 0)

with netCDF4.Dataset(tmp_local_or_remote_netcdf, "a") as f:
    f.createVariable("dummy", "i8", ("z", "y"))
    f.createVariable("z", "i8", ("x", "y"))
    f["dummy"][:] = np.ones((3,1))
    f["z"][:] = np.ones((2,1))
    #f.resize_dimension("z", 3)


# # Assert some behavior observed by using the C netCDF bindings.
# with h5py.File(tmp_local_or_remote_netcdf, "r") as f:
#     assert f["x"].shape == (0,)
#     assert f["x"].maxshape == (None,)
#     assert f["y"].shape == (1,)
#     assert f["y"].maxshape == (1,)
#     assert f["z"].shape == (2, 1,)
#     assert f["z"].maxshape == (None, 1,)

with h5netcdf.File(tmp_local_or_remote_netcdf, "r") as f:
    print("----------------------------------------")
    print(f.dimensions)
    print("h5netcdf variables:", list(f.variables))
    print("h5netcdf dummy:", f.variables["dummy"])
    print("h5netcdf z:", f.variables["z"])

In [ ]:
!ncdump test_dims_netcdf4.nc

In [ ]:
!h5dump test_dims_netcdf4.nc

In [ ]:
tmp_local_or_remote_netcdf = "test_dims_h5netcdf.nc"
with h5netcdf.File(tmp_local_or_remote_netcdf, "w") as f:
    f.dimensions["x"] = None
    f.dimensions["y"] = 1
    f.dimensions["z"] = None
with h5netcdf.File(tmp_local_or_remote_netcdf, "a") as f:
    f.resize_dimension("z", 1)
    f.create_variable("dummy", dimensions=("z", "y"), dtype=np.int64)
    f.create_variable("z", dimensions=("x", "y"), dtype=np.int64)
    f.resize_dimension("z", 3)

    with pytest.raises(ValueError) as e:
        f.resize_dimension("y", 20)
    assert e.value.args[0] == (
        "Dimension 'y' is not unlimited and thus cannot be resized."
    )

# # Assert some behavior observed by using the C netCDF bindings.
# with h5py.File(tmp_local_or_remote_netcdf, "r") as f:
#     assert f["x"].shape == (0,)
#     assert f["x"].maxshape == (None,)
#     assert f["y"].shape == (1,)
#     assert f["y"].maxshape == (1,)
#     assert f["z"].shape == (2, 1,)
#     assert f["z"].maxshape == (None, 1,)

with h5netcdf.File(tmp_local_or_remote_netcdf, "r") as f:
    print("----------------------------------------")
    print(f.dimensions)
    print("h5netcdf variables:", list(f.variables))
    print("h5netcdf dummy:", f.variables["dummy"])
    print("h5netcdf z:", f.variables["z"])

In [ ]:
!ncdump test_dims.nc

In [ ]:
!h5dump test_dims.nc